# Task 3 - Multi-Agent Financial Research System

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indeewara/CDAZZDEV-MLE-IndeewaraJayasuriya/blob/main/task3_agentic/task3_multi_agent_research.ipynb)

Agents that research a stock with five tools, decide each step from what they observe, hand structured data to each
other, remember context, and log every tool call.

| Part | Built with | Code |
|---|---|---|
| **3A** Tool-using research agent | LangGraph loop (agent ⇄ tools), `gpt-oss-120b` on Groq, one tool call per step | `tools.py`, `agent.py` |
| **3B** Two-agent pipeline | Data Analyst + Research Writer with enforced tool access, Pydantic hand-offs, critique loop | `multi_agent.py` |
| **3C** Memory and observability | LangGraph `MemorySaver` (short-term), JSON cache keyed by ticker+date (persistent), `logs/agent_trace.jsonl` | `memory.py`, `tracing.py` |

**Run:** locally the Groq key comes from `.env`; on Colab add it as the Colab Secret `GROQ_API_KEY`. No GPU needed.
Groq's free tier gives each model its own daily token quota (200k for `gpt-oss-120b`); a full run needs ~130k. If a
model's quota runs out mid-run, the agents move along a chain - `gpt-oss-120b` → `qwen3.8-27b` → `gpt-oss-20b` - and
every switch is logged in the trace (`model_fallback` events), so the outputs below show which model produced them.

## 0. Setup

In [1]:
import os, subprocess, sys

REPO_URL = "https://github.com/indeewara/CDAZZDEV-MLE-IndeewaraJayasuriya.git"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def sh(cmd):
    # run a command and SHOW its output if it fails - Colab otherwise hides subprocess output
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:])
        print(r.stderr[-3000:])
        raise RuntimeError("command failed: " + " ".join(cmd[:6]))
    return r

if IN_COLAB:
    if not os.path.exists("CDAZZDEV-MLE-IndeewaraJayasuriya"):
        sh(["git", "clone", "-q", REPO_URL])
    else:
        sh(["git", "-C", "CDAZZDEV-MLE-IndeewaraJayasuriya", "pull", "-q"])
    os.chdir("CDAZZDEV-MLE-IndeewaraJayasuriya/task3_agentic")
    sh([sys.executable, "-m", "pip", "install", "-q", "langgraph==1.2.14", "langchain-groq==1.1.3",
        "langchain-core==1.6.7", "ddgs==9.16.0", "yfinance", "feedparser", "groq", "python-dotenv", "pydantic"])
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

sys.path.insert(0, os.getcwd())
commit = subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print("working dir: .../" + os.path.basename(os.getcwd()), "| Colab:", IN_COLAB, "| code version:", commit)

working dir: .../task3_agentic | Colab: False | code version: d095772 CITATIONS.md: clearer task-level prompts


In [2]:
import json, logging
import pandas as pd
from IPython.display import Markdown, display

logging.disable(logging.WARNING)          # silence yfinance/library chatter; the agent trace is printed below
pd.set_option("display.max_colwidth", 90)

import tracing
from tools import TOOLS
from agent import AGENT_MODEL, MAX_TOOL_ROUNDS, rationale, run_research
from multi_agent import DATA_ANALYST_TOOLS, RESEARCH_WRITER_TOOLS, run_pipeline
from memory import ask_followup, cache_path, load_report, research_with_cache, save_report

tracing.VERBOSE = True
print("agent model:", AGENT_MODEL, "| fallback chain:", " -> ".join(tracing.MODEL_ORDER), "| tool budget:", MAX_TOOL_ROUNDS)
print("trace file:", tracing.TRACE_PATH.relative_to(tracing.ROOT))

def trace(session=None):
    rows = [json.loads(l) for l in tracing.TRACE_PATH.read_text().splitlines()]
    return [r for r in rows if session is None or r.get("session_id") == session]

agent model: openai/gpt-oss-120b | fallback chain: openai/gpt-oss-120b -> qwen/qwen3.8-27b -> openai/gpt-oss-20b | tool budget: 8
trace file: task3_agentic/logs/agent_trace.jsonl


---
# 3A - Tool-Using Research Agent

## The five tools
Each tool returns a compact JSON-serialisable `dict`; a failure returns `{"error": ...}` instead of raising, so the
agent can observe it and try something else. Price, indicator and news code is reused from Task 1 (already tested).

In [3]:
for name, t in TOOLS.items():
    print(f"{name}({', '.join(t.args)})\n    {t.description.splitlines()[0]}")

get_price_data(ticker, period)
    Daily OHLCV price history for a stock with computed technical indicators (SMA 50/200, RSI 14, MACD,
calculate_volatility(ticker, window)
    Annualised historical volatility of daily log returns over a trailing window of trading days, compared with
get_news(ticker, n)
    Recent news headlines for a stock (title, publisher, date), newest first.
llm_sentiment(headlines)
    Scores a list of news headline strings with an LLM (positive / negative / neutral with confidence) and
web_search(query, max_results)
    Web search (DuckDuckGo) for analyst commentary, price targets, upcoming events and risks not covered by


Each tool called once directly, to show it is callable and what type it returns:

In [4]:
tracing.new_session()
samples = {
    "get_price_data": {"ticker": "AAPL", "period": "1y"},
    "calculate_volatility": {"ticker": "AAPL", "window": 30},
    "get_news": {"ticker": "AAPL", "n": 5},
    "web_search": {"query": "Apple stock analyst price target"},
}
outputs = {name: TOOLS[name].invoke(args) for name, args in samples.items()}
outputs["llm_sentiment"] = TOOLS["llm_sentiment"].invoke({"headlines": [h["title"] for h in outputs["get_news"]["headlines"]]})
display(pd.DataFrame([{"tool": n, "returns": type(o).__name__, "keys": ", ".join(list(o)[:7]),
                       "error": o.get("error")} for n, o in outputs.items()]))

    [research_agent] tool get_price_data(ticker='AAPL', period='1y') -> ok in 1409 ms: {"ticker": "AAPL", "company": "Apple Inc.", "period": "1y", "rows": 251, "period_return_pct": 30.56, "current_price": 333.63, "high_52w": 345.34, "low...


    [research_agent] tool calculate_volatility(ticker='AAPL', window=30) -> ok in 782 ms: {"ticker": "AAPL", "window_days": 30, "annualised_vol_pct": 21.06, "one_year_avg_vol_pct": 24.17, "one_year_range_vol_pct": [10.29, 41.15], "percentil...


    [research_agent] tool get_news(ticker='AAPL', n=5) -> ok in 2613 ms: {"ticker": "AAPL", "count": 5, "headlines": [{"title": "Apple (AAPL) Shares Enter 24 7 Token Trading Plan In 63 Stock Filing", "publisher": "Simply Wa...


    [research_agent] tool web_search(query='Apple stock analyst price target', max_results=5) -> ok in 6495 ms: {"query": "Apple stock analyst price target", "results": [{"title": "Apple (AAPL) Stock Forecast and Price Target 2026 - MarketBeat", "snippet": "AAPL...


    [research_agent] tool llm_sentiment(headlines=['Apple (AAPL) Shares Enter 24 7 Token Trading Plan In 63 Stock Filing) -> ok in 1390 ms: {"aggregate": {"score": 0.0256, "label": "neutral", "counts": {"positive": 1, "neutral": 3, "negative": 1}, "n_scored": 5, "n_failed": 0}, "per_headli...


,tool,returns,keys,error
0,get_price_data,dict,"ticker, company, period, rows, period_return_pct, current_price, high_52w",None
1,calculate_volatility,dict,"ticker, window_days, annualised_vol_pct, one_year_avg_vol_pct, one_year_range_vol_pct,...",None
2,get_news,dict,"ticker, count, headlines",None
3,web_search,dict,"query, results",None
4,llm_sentiment,dict,"aggregate, per_headline",None


## Autonomous research run
The query from the brief. The agent sees the tools' descriptions and decides every step itself - there is no fixed
order in the code. Each step prints what the model was thinking (`thinks`), the tool it chose (`decides`) and the
result it observed; then the report writer turns the observations into the three required sections and checks every
number in the report against the tool results.

In [5]:
report_a, run_a = run_research("AAPL")

session 7d964d96 | question: Analyse the current financial health and market sentiment of AAPL. Identify the top three risks to its share price over the next 90 days and suggest one data-driven hedge strategy.



[research_agent] thinks: We need data. Call get_price_data.Observed no data yet; need price and fundamentals for AAPL.
[research_agent] decides: call get_price_data({"period": "1y", "ticker": "AAPL"})


    [research_agent] tool get_price_data(ticker='AAPL', period='1y') -> ok in 802 ms: {"ticker": "AAPL", "company": "Apple Inc.", "period": "1y", "rows": 251, "period_return_pct": 30.56, "current_price": 333.63, "high_52w": 345.34, "low...


[research_agent] thinks: We have price data. Next get volatility.Having seen a strong YTD return and neutral momentum, I’ll calculate Apple’s recent volatility to gauge price swing risk over the next 90 days.
[research_agent] decides: call calculate_volatility({"ticker": "AAPL", "window": 30})


    [research_agent] tool calculate_volatility(ticker='AAPL', window=30) -> ok in 670 ms: {"ticker": "AAPL", "window_days": 30, "annualised_vol_pct": 21.06, "one_year_avg_vol_pct": 24.17, "one_year_range_vol_pct": [10.29, 41.15], "percentil...
[research_agent] openai/gpt-oss-120b daily quota used up - falling back to qwen/qwen3.8-27b


...

[research_agent] qwen/qwen3.8-27b daily quota used up - falling back to openai/gpt-oss-20b


[research_agent] thinks: We have data: price, volatility, sentiment neutral, some negative headlines about memory costs, insider trading. Need top 3 risks: 1) Memory cost impact on margins, 2) App Store growth slowdown, 3) China tariff/regulatory risk. Hedge: use Apple inverse ETF or short-term options? Data-driven: use 90d...
[research_agent] answers: **Financial health snapshot (1‑year)**  
| Metric | Value | Comment |
|--------|-------|---------|
| Current price | **$333.63** | Near the 52‑week high ($345.34) |
| 52‑week range | $242.76 – $345.34 | 30 % upside potential |
| P/E | **38.3** | Above the S&P 500 average – valuation pressure |
| YTD return | **+23 %** | Strong performance, but momentum is “neutral” (MACD below signal, RSI ~55) |
| 1‑year volatility | **21 %** | 29th percentile vs. last year (24 % avg) – relatively low |
| Expected 90‑day 1‑σ move | **±12.6 %** → $291.6 – $375.6 | 90‑day downside risk ≈ $42 |

**Market sentimen


[report_writer] grounding check 1 (ResearchReport): numbers not in observations: ['500']


[report_writer] grounding check 2 (ResearchReport): numbers not in observations: ['500']


[report_writer] grounding check 3 (ResearchReport): numbers not in observations: ['500']
[report_writer] WARNING: after 3 attempts these numbers are still not traceable to a tool result: ['500'] - treat them as unverified


In [6]:
display(Markdown(report_a.to_markdown()))
print("numbers in the report not traceable to a tool result:", run_a["ungrounded"] or "none")

# Research report: AAPL

## Financial Health Summary
Apple’s current price of 333.63 sits near the 52‑week high of 345.34, giving a 30.56% 1‑year return and a 23.06% YTD return. The P/E of 38.3 is above the S&P 500 average, indicating valuation pressure. Momentum is neutral with a MACD below signal and RSI around 54.6, while volatility is 21.06% annualised, below the 24.17% one‑year average and in the 29th percentile. The expected 90‑day 1‑σ move is ±12.58%, implying a downside of 291.64 and an upside of 375.62.

## Top Three Risks (next 90 days)
1. **Memory‑cost margin squeeze** (high)
   Evidence: Apple (AAPL) Trading Above Analyst Targets as Memory Costs Crimp Margin Outlook headline indicates rising memory costs could erode margins
2. **App Store growth slowdown** (medium)
   Evidence: UBS reiterates Apple stock neutral on slowing App Store growth headline signals potential revenue drag
3. **China tariff / regulatory risk** (medium)
   Evidence: Multiple news items reference U.S.–China tariffs and their impact on iPhone pricing and supply chain

## Hedge Strategy Recommendation
**Buy 90‑day out‑of‑the‑money put** - AAPL 90‑day put at strike 330
Sizing: one put contract per 100 shares held
Rationale: The 90‑day expected 1‑σ downside is 12.58%, equating to a price drop to 291.64. A 330 strike put provides protection near the current price while allowing upside participation above 330, aligning with the observed volatility and price band.

*Sources: get_price_data, calculate_volatility, get_news, llm_sentiment*

numbers in the report not traceable to a tool result: ['500']


### Observe → decide, step by step
Each row is one decision: the agent's own reason at that point (its stated sentence, or its reasoning text for
models that return it), the call it made, and what it observed.

In [7]:
from langchain_core.messages import AIMessage, ToolMessage

def decision_table(messages):
    rows, step = [], 0
    for m in messages:
        if isinstance(m, AIMessage) and m.tool_calls:
            step += 1
            c = m.tool_calls[0]
            rows.append({"step": step, "why (agent's own words)": rationale(m)[:150],
                         "decided to call": f"{c['name']}({json.dumps(c['args'])[:70]})"})
        elif isinstance(m, ToolMessage) and rows:
            rows[-1]["observed"] = ("ERROR: " if '"error"' in m.content[:12] else "") + m.content[:90]
    return pd.DataFrame(rows).set_index("step")

display(decision_table(run_a["state"]["messages"]))

,why (agent's own words),decided to call,observed
step,,,
1,We need data. Call get_price_data.Observed no data yet; need price and fundamentals fo...,"get_price_data({""period"": ""1y"", ""ticker"": ""AAPL""})","{""ticker"": ""AAPL"", ""company"": ""Apple Inc."", ""period"": ""1y"", ""rows"": 251, ""period_retur..."
2,We have price data. Next get volatility.Having seen a strong YTD return and neutral mo...,"calculate_volatility({""ticker"": ""AAPL"", ""window"": 30})","{""ticker"": ""AAPL"", ""window_days"": 30, ""annualised_vol_pct"": 21.06, ""one_year_avg_vol_p..."
3,,"get_news({""n"": 10, ""ticker"": ""AAPL""})","{""ticker"": ""AAPL"", ""count"": 10, ""headlines"": [{""title"": ""Apple (AAPL) Shares Enter 24 ..."
4,,"llm_sentiment({""headlines"": [""Apple (AAPL) Shares Enter 24 7 Token Trading Plan In 6)","{""aggregate"": {""score"": -0.0988, ""label"": ""neutral"", ""counts"": {""positive"": 1, ""neutra..."
5,,"web_search({""max_results"": 5, ""query"": ""AAPL Apple stock analyst price target ris)","{""query"": ""AAPL Apple stock analyst price target risk next 90 days memory costs App St..."
6,,"web_search({""max_results"": 5, ""query"": ""Apple AAPL China demand risk tariff iPhon)","{""query"": ""Apple AAPL China demand risk tariff iPhone 2026"", ""results"": [{""title"": ""iP..."


## Error handling: a tool outage
`get_news` is forced to fail for this run (`tracing.FAILURE_INJECTION`). The agent is not told what to do about it;
it has to notice the error and find another way to get the information.

In [8]:
tracing.FAILURE_INJECTION.add("get_news")
try:
    report_err, run_err = run_research("NVDA")
finally:
    tracing.FAILURE_INJECTION.clear()

session c41ef26a | question: Analyse the current financial health and market sentiment of NVDA. Identify the top three risks to its share price over the next 90 days and suggest one data-driven hedge strategy.



[research_agent] thinks: Need price data, volatility, news, sentiment, web search.
[research_agent] decides: call get_price_data({"period": "1y", "ticker": "NVDA"})


    [research_agent] tool get_price_data(ticker='NVDA', period='1y') -> ok in 623 ms: {"ticker": "NVDA", "company": "NVIDIA Corporation", "period": "1y", "rows": 251, "period_return_pct": 29.6, "current_price": 239.24, "high_52w": 243.3...


[research_agent] thinks: Need volatility, news sentiment, web search.
[research_agent] decides: call calculate_volatility({"ticker": "NVDA", "window": 30})


    [research_agent] tool calculate_volatility(ticker='NVDA', window=30) -> ok in 235 ms: {"ticker": "NVDA", "window_days": 30, "annualised_vol_pct": 37.42, "one_year_avg_vol_pct": 37.8, "one_year_range_vol_pct": [26.66, 47.44], "percentile...


...

    [research_agent] tool web_search(query='NVDA analyst price target 2026 October', max_results=5) -> ok in 2689 ms: {"query": "NVDA analyst price target 2026 October", "results": [{"title": "NVDA Stock Forecast & Price Target — Oct 2026", "snippet": "60 analysts cov...


[research_agent] thinks: Need risks and hedge. Use volatility, trend, sentiment neutral. Risks: 1) Overbought RSI and near upper Bollinger, risk of pullback. 2) High valuation P/E 30.2 vs peers, earnings miss. 3) Macro tech slowdown or supply chain. Hedge: short-term options or inverse ETF. Suggest buying NVDA put options a...
[research_agent] answers: **Financial health snapshot (as of Oct 6 2026)**  

| Metric | Value | Context |
|--------|-------|---------|
| Current price | **$239.24** | Near the 52‑week high ($243.37) |
| 52‑week range | $163.9 – $243.37 | 16 % of the range is already hit |
| P/E | **30.2** | Above the semiconductor average (~25) but still within the 2024‑2026 range |
| YTD return | **+28.6 %** | Strong performance, but momentum is slowing (RSI 67.3, MACD above signal) |
| Trend | Bullish (SMA‑50 above SMA‑200, golden‑cross regime) | Still bullish but near upper Bollinger Band |
| Volatility | **37.4 %** (30‑day window)


[report_writer] grounding check 1 (ResearchReport): numbers not in observations: ['40', '215', '350']


[report_writer] grounding check 2 (ResearchReport): numbers not in observations: ['37']


[report_writer] grounding check 3 (ResearchReport): numbers not in observations: ['40', '215', '350']
[report_writer] WARNING: after 3 attempts these numbers are still not traceable to a tool result: ['40', '215', '350'] - treat them as unverified


In [9]:
display(decision_table(run_err["state"]["messages"]))
print("report produced despite the outage:", report_err is not None,
      "| sources used:", report_err.data_sources if report_err else None)

,why (agent's own words),decided to call,observed
step,,,
1,"Need price data, volatility, news, sentiment, web search.","get_price_data({""period"": ""1y"", ""ticker"": ""NVDA""})","{""ticker"": ""NVDA"", ""company"": ""NVIDIA Corporation"", ""period"": ""1y"", ""rows"": 251, ""peri..."
2,"Need volatility, news sentiment, web search.","calculate_volatility({""ticker"": ""NVDA"", ""window"": 30})","{""ticker"": ""NVDA"", ""window_days"": 30, ""annualised_vol_pct"": 37.42, ""one_year_avg_vol_p..."
3,Need news sentiment.,"get_news({""n"": 10, ""ticker"": ""NVDA""})","ERROR: {""error"": ""get_news unavailable (simulated outage for the error-handling demo)""}"
4,Need alternative: web_search for news.,"web_search({""max_results"": 5, ""query"": ""NVDA recent news headlines""})","{""query"": ""NVDA recent news headlines"", ""results"": [{""title"": ""NVIDIA Corporation (NVD..."
5,Need sentiment. Use llm_sentiment on headlines from web search? We have titles. Use th...,"llm_sentiment({""headlines"": [""NVIDIA Corporation (NVDA) Latest Stock News & Headline)","{""aggregate"": {""score"": 0.0, ""label"": ""neutral"", ""counts"": {""positive"": 0, ""neutral"": ..."
6,We lack specific news. Need analyst commentary. Search for analyst price targets.,"web_search({""max_results"": 5, ""query"": ""NVDA analyst price target 2026 October""})","{""query"": ""NVDA analyst price target 2026 October"", ""results"": [{""title"": ""NVDA Stock ..."


report produced despite the outage: True | sources used: ['get_price_data', 'calculate_volatility']


---
# 3B - Multi-Agent Coordination

| Agent | Role | Tools (enforced) | Output |
|---|---|---|---|
| **A - Data Analyst** | quantitative analysis | get_price_data, calculate_volatility, llm_sentiment | `DataBrief` (Pydantic) |
| **B - Research Writer** | qualitative synthesis | web_search, get_news | `MultiAgentReport` (Pydantic) |

Neither agent can produce news sentiment alone: B has the headlines but no sentiment tool; A has the sentiment tool
but cannot fetch news. Tool access is enforced in the graph - a call to another agent's tool is refused - not just
stated in the prompt.

```
A: price + volatility --DataBrief--> B: news + web search --ClarificationRequest--> A: llm_sentiment
                                     B: final report <--ClarificationResponse-------/
```

**Enforcement check** (a scripted model tries to use the other agent's tool; no API call):

In [10]:
from langchain_core.messages import HumanMessage
from agent import build_agent

class ScriptedLLM:
    def __init__(self, replies): self.replies = list(replies)
    def bind_tools(self, tools, **_): return self
    def invoke(self, messages): return self.replies.pop(0)

def attempt(agent_name, tool_names, forbidden):
    llm = ScriptedLLM([AIMessage(content="", tool_calls=[{"name": forbidden, "args": {"ticker": "AAPL"}, "id": "x", "type": "tool_call"}]),
                       AIMessage(content="done")])
    g = build_agent([TOOLS[n] for n in tool_names], "sys {max_rounds}", name=agent_name, llm=llm)
    out = g.invoke({"messages": [HumanMessage("go")], "tool_rounds": 0}, {"recursion_limit": 20})
    return next(m.content for m in out["messages"] if isinstance(m, ToolMessage))

tracing.VERBOSE = False
print("Research Writer tries get_price_data ->", attempt("research_writer", RESEARCH_WRITER_TOOLS, "get_price_data"))
print("Data Analyst tries get_news         ->", attempt("data_analyst", DATA_ANALYST_TOOLS, "get_news"))
tracing.VERBOSE = True

Research Writer tries get_price_data -> {"error": "tool 'get_price_data' is not available to research_writer; available: ['get_news', 'web_search']"}
Data Analyst tries get_news         -> {"error": "tool 'get_news' is not available to data_analyst; available: ['calculate_volatility', 'get_price_data', 'llm_sentiment']"}


## Full pipeline run - the complete message trace
Everything below is printed live: each agent's reasoning, decisions and tool calls, and every hand-off as the
validated JSON object that was passed.

In [11]:
report_b, run_b = run_pipeline("MSFT")

session 3459f5c5 | Analyse the current financial health and market sentiment of MSFT. Identify the top three risks to its share price over the next 90 days and suggest one data-driven hedge strategy.

================== 1. Agent A - Data Analyst researches the numbers ==================


[data_analyst] thinks: Need price data 1y.
[data_analyst] decides: call get_price_data({"period": "1y", "ticker": "MSFT"})


    [data_analyst] tool get_price_data(ticker='MSFT', period='1y') -> ok in 1070 ms: {"ticker": "MSFT", "company": "Microsoft Corporation", "period": "1y", "rows": 251, "period_return_pct": 1.85, "current_price": 529.3, "high_52w": 549...


[data_analyst] thinks: Need volatility.
[data_analyst] decides: call calculate_volatility({"ticker": "MSFT", "window": 30})


    [data_analyst] tool calculate_volatility(ticker='MSFT', window=30) -> ok in 348 ms: {"ticker": "MSFT", "window_days": 30, "annualised_vol_pct": 22.08, "one_year_avg_vol_pct": 30.06, "one_year_range_vol_pct": [13.97, 54.03], "percentil...


...

[data_analyst] thinks: Need to call llm_sentiment.
[data_analyst] decides: call llm_sentiment({"headlines": ["Jim Cramer Says Microsoft\u2019s (MSFT) AI Investment Is Finally Paying Off", "Is Azure Still Driving Microsoft (NASDAQ:MSFT) Higher?", "Microso)


    [data_analyst] tool llm_sentiment(headlines=['Jim Cramer Says Microsoft’s (MSFT) AI Investment Is Finally Paying O) -> ok in 52248 ms: {"aggregate": {"score": 0.0125, "label": "neutral", "counts": {"positive": 2, "neutral": 1, "negative": 2}, "n_scored": 5, "n_failed": 0}, "per_headli...


[data_analyst] answers: **Sentiment Analysis – MSFT Headlines**

| # | Headline | Sentiment | Confidence |
|---|----------|-----------|------------|
| 1 | Jim Cramer Says Microsoft’s (MSFT) AI Investment Is Finally Paying Off | **Positive** | 0.90 |
| 2 | Is Azure Still Driving Microsoft (NASDAQ:MSFT) Higher? | **Neutral** | 0.55 |
| 3 | Microsoft Stock Is on Track to Trail the S&P 500 for a 3rd Straight Year. History Says What Happened After Its Last 2 Streaks. | **Negative** | 0.80 |
| 4 | MSFT Stock Nears All‑Time High: Xbox Moves To Secure Exclusive Rights To Stream Grand Theft Auto VI, Report Says | **Positive**


[data_analyst] grounding check 1 (ClarificationResponse): every number traced to a tool result

>>> HAND-OFF data_analyst -> research_writer [ClarificationResponse]
{
  "answer": "The five MSFT headlines received a mixed sentiment: two were positive, one neutral, and two negative, yielding an aggregate score of +0.0125 labeled neutral. The confidence levels ranged from 0.55 for the neutral headline to 0.90 for the two negative headlines. Overall, the media tone is slightly neutral with an even split between positive and negative coverage.",
  "key_figures": {
    "aggregate_score": "+0.0125",
    "positive_count": "2",
    "neutral_count": "1",
    "negative_count": "2"
  },
  "tools_used": [
    "llm_sentiment"
  ]
}

================== 5. Agent B writes the final report ==================


[research_writer] grounding check 1 (MultiAgentReport): every number traced to a tool result
[research_writer] Data Analyst's answer used in the report's analysis: YES


In [12]:
display(Markdown(report_b.to_markdown()))
print("numbers not traceable to a tool result:", run_b["ungrounded"] or "none",
      "| Data Analyst's answer used in the analysis:", run_b["incorporated"])

# Research report: MSFT

## Financial Health Summary
The current price of 529.3 sits above both the 50‑day SMA (493.09) and the 200‑day SMA (432.0), confirming a bullish trend and a golden‑cross regime. The RSI of 68.01 is approaching overbought territory, suggesting potential short‑term pullback risk. Annualised volatility of 22.08% is below the one‑year average of 30.06%, indicating a relatively calm period compared to the past year, yet the price lies within the 34th percentile of its volatility range, so volatility is not exceptionally low.

## Top Three Risks (next 90 days)
1. **Overbought RSI and potential pullback** (medium)
   Evidence: RSI of 68.01 is approaching overbought territory
2. **Regulatory scrutiny from U.S. antitrust regulators** (high)
   Evidence: Reuters article on U.S. FTC antitrust probe into Microsoft’s software licensing and cloud computing businesses
3. **Mixed media sentiment and potential short‑term volatility** (low)
   Evidence: Data Analyst sentiment: aggregate score +0.0125, two negative headlines

## Hedge Strategy Recommendation
**Buy protective puts at 1‑sigma lower bound** - MSFT put options
Sizing: one put contract per 100 shares held
Rationale: The expected 90‑day move is 13.19%, with a 1‑sigma lower price of 459.46. Purchasing puts near this level provides downside protection against a potential pullback while keeping cost aligned with the 90‑day volatility forecast.

*Sources: web_search, get_news, data_analyst*

## How the Data Analyst's clarification was used
I asked the Data Analyst to score the sentiment of five recent MSFT headlines. The analyst returned an aggregate neutral score of +0.0125 with two negative headlines, which informed the risk assessment of mixed media sentiment and its potential impact on short‑term volatility.

numbers not traceable to a tool result: none | Data Analyst's answer used in the analysis: True


Hand-offs for this run, read back from `agent_trace.jsonl`:

In [13]:
display(pd.DataFrame([{"from": e["from"], "to": e["to"], "schema": e["schema"],
                       "payload (start)": json.dumps(e["payload"])[:110]}
                      for e in trace(run_b["session_id"]) if e["event"] == "handoff"]))
by_agent = {}
for e in trace(run_b["session_id"]):
    if e["event"] == "tool_call":
        by_agent.setdefault(e["agent"], []).append(e["tool"])
print("tools actually called per agent:", by_agent)

,from,to,schema,payload (start)
0,data_analyst,research_writer,DataBrief,"{""ticker"": ""MSFT"", ""current_price"": 529.3, ""high_52w"": 549.2, ""low_52w"": 348.54, ""sma_..."
1,research_writer,data_analyst,ClarificationRequest,"{""to_agent"": ""data_analyst"", ""question"": ""Please score the sentiment of the following ..."
2,data_analyst,research_writer,ClarificationResponse,"{""answer"": ""The five MSFT headlines received a mixed sentiment: two were positive, one..."


tools actually called per agent: {'data_analyst': ['get_price_data', 'calculate_volatility', 'llm_sentiment'], 'research_writer': ['web_search', 'get_news', 'web_search']}


---
# 3C - Memory and Observability

## Short-term memory
A follow-up question sent into the same conversation thread as the 3A run. The tool calls made while answering are
counted from the new messages and from the trace: **0 means it was answered from memory**.

In [14]:
q = ("What 30-day annualised volatility and 90-day downside price band did you find, "
     "and what was the headline sentiment score?")
followup = ask_followup(run_a["graph"], run_a["config"], q)
print("Q:", q, "\n")
print("A:", followup["answer"], "\n")
print("messages already in memory:", followup["messages_in_memory_before"],
      "| tool calls in new messages:", followup["tool_calls_in_messages"] or 0,
      "| tool calls in trace:", followup["tool_calls_in_trace"])

[research_agent] thinks: Need to answer from earlier results.
[research_agent] answers: - **30‑day annualised volatility**: **21.06 %** (lower than the 24.17 % average for the past year).  
- **90‑day downside price band**: **$291.6** (≈ 12.6 % below the current price of $333.63).  
- **Headline sentiment score**: **‑0.0988** (neutral overall, with 1 positive, 7 neutral, and 2 negative headlines).
Q: What 30-day annualised volatility and 90-day downside price band did you find, and what was the headline sentiment score? 

A: - **30‑day annualised volatility**: **21.06 %** (lower than the 24.17 % average for the past year).  
- **90‑day downside price band**: **$291.6** (≈ 12.6 % below the current price of $333.63).  
- **Headline sentiment score**: **‑0.0988** (neutral overall, with 1 positive, 7 neutral, and 2 negative headlines). 

messages already in memory: 14 | tool calls in new messages: 0 | tool calls in trace: 0


## Persistent memory
The 3B session's final brief is saved to `cache/<TICKER>_<date>.json`. A second request for the same ticker on the
same day detects the file and loads it instead of running any tool.

In [15]:
path = save_report(report_b, run_b["session_id"])
print("saved:", path.relative_to(tracing.ROOT))
print(json.dumps(json.loads(path.read_text()), indent=2)[:700], "...")

saved: task3_agentic/cache/MSFT_2026-10-07.json
{
  "ticker": "MSFT",
  "date": "2026-10-07",
  "created_at": "2026-10-07T09:36:23+00:00",
  "session_id": "3459f5c5",
  "report_type": "MultiAgentReport",
  "report": {
    "ticker": "MSFT",
    "financial_health_summary": "The current price of 529.3 sits above both the 50\u2011day SMA (493.09) and the 200\u2011day SMA (432.0), confirming a bullish trend and a golden\u2011cross regime. The RSI of 68.01 is approaching overbought territory, suggesting potential short\u2011term pullback risk. Annualised volatility of 22.08% is below the one\u2011year average of 30.06%, indicating a relatively calm period compared to the past year, yet the price lies within the 34th percentile of its volatility ...


In [16]:
cached, info = research_with_cache("MSFT", run_pipeline)   # would run the whole pipeline on a cache miss
print("source:", info["source"], "| tool calls:", info["tool_calls"], "| seconds:", info["seconds"],
      "| same report:", cached == report_b)

cache HIT: loaded MSFT_2026-10-07.json - no tools run (0 tool calls, 0.001 s)
source: cache | tool calls: 0 | seconds: 0.001 | same report: True


## Observability: `logs/agent_trace.jsonl`
One JSON line per event. Every tool call records the tool, its inputs, its output truncated to 200 characters, the
wall-clock duration, its status and the calling agent; hand-offs, grounding checks, cache and model-fallback events
are logged too.

In [17]:
events = trace()
print(f"{len(events)} events in {tracing.TRACE_PATH.name}:",
      dict(pd.Series([e['event'] for e in events]).value_counts()))
calls = pd.DataFrame([e for e in events if e["event"] == "tool_call"])
assert (calls["output"].str.len() <= tracing.MAX_OUTPUT_CHARS).all()
display(calls[["ts", "agent", "tool", "inputs", "output", "duration_ms", "status"]].tail(12))
display(calls.groupby("tool")["duration_ms"].describe()[["count", "mean", "max"]].round(0))

227 events in agent_trace.jsonl: {'tool_call': np.int64(109), 'grounding_check': np.int64(48), 'handoff': np.int64(15), 'model_fallback': np.int64(8), 'tool_refused': np.int64(8), 'pipeline_start': np.int64(5), 'pipeline_end': np.int64(5), 'cache_saved': np.int64(5), 'followup_question': np.int64(5), 'followup_answer': np.int64(4), 'cache_hit': np.int64(4), 'critique_incorporated': np.int64(3), 'rate_limit_wait': np.int64(3), 'cache_miss': np.int64(2), 'llm_unavailable': np.int64(2), 'validation_error': np.int64(1)}


,ts,agent,tool,inputs,output,duration_ms,status
97,2026-10-07T09:28:04.555+00:00,research_agent,get_price_data,"{'ticker': 'NVDA', 'period': '1y'}","{""ticker"": ""NVDA"", ""company"": ""NVIDIA Corporation"", ""period"": ""1y"", ""rows"": 251, ""peri...",623.4,ok
98,2026-10-07T09:28:05.483+00:00,research_agent,calculate_volatility,"{'ticker': 'NVDA', 'window': 30}","{""ticker"": ""NVDA"", ""window_days"": 30, ""annualised_vol_pct"": 37.42, ""one_year_avg_vol_p...",235.1,ok
99,2026-10-07T09:29:04.855+00:00,research_agent,get_news,"{'ticker': 'NVDA', 'n': 10}","{""error"": ""get_news unavailable (simulated outage for the error-handling demo)""}",0.0,error
100,2026-10-07T09:29:09.984+00:00,research_agent,web_search,"{'query': 'NVDA recent news headlines', 'max_results': 5}","{""query"": ""NVDA recent news headlines"", ""results"": [{""title"": ""NVIDIA Corporation (NVD...",4636.2,ok
101,2026-10-07T09:29:11.623+00:00,research_agent,llm_sentiment,"{'headlines': ['NVIDIA Corporation (NVDA) Latest Stock News & Headlines ...', 'Nvidia:...","{""aggregate"": {""score"": 0.0, ""label"": ""neutral"", ""counts"": {""positive"": 0, ""neutral"": ...",875.0,ok
102,2026-10-07T09:30:09.964+00:00,research_agent,web_search,"{'query': 'NVDA analyst price target 2026 October', 'max_results': 5}","{""query"": ""NVDA analyst price target 2026 October"", ""results"": [{""title"": ""NVDA Stock ...",2689.0,ok
103,2026-10-07T09:32:16.369+00:00,data_analyst,get_price_data,"{'ticker': 'MSFT', 'period': '1y'}","{""ticker"": ""MSFT"", ""company"": ""Microsoft Corporation"", ""period"": ""1y"", ""rows"": 251, ""p...",1070.4,ok
104,2026-10-07T09:32:17.225+00:00,data_analyst,calculate_volatility,"{'ticker': 'MSFT', 'window': 30}","{""ticker"": ""MSFT"", ""window_days"": 30, ""annualised_vol_pct"": 22.08, ""one_year_avg_vol_p...",348.2,ok
105,2026-10-07T09:33:19.071+00:00,research_writer,web_search,"{'query': 'Microsoft MSFT analyst commentary price target 2024 Q3 risks', 'max_results...","{""query"": ""Microsoft MSFT analyst commentary price target 2024 Q3 risks"", ""results"": [...",3105.7,ok
106,2026-10-07T09:33:20.976+00:00,research_writer,get_news,"{'ticker': 'MSFT', 'n': 5}","{""ticker"": ""MSFT"", ""count"": 5, ""headlines"": [{""title"": ""Jim Cramer Says Microsoft’s (M...",1532.3,ok


,count,mean,max
tool,,,
calculate_volatility,20.0,432.0,1341.0
get_news,20.0,1537.0,4031.0
get_price_data,20.0,1237.0,2248.0
llm_sentiment,17.0,4307.0,52248.0
web_search,32.0,4102.0,10034.0


---
# Tests
Offline (no API calls): a scripted model drives the real agent graph to test the observe/decide loop, tool refusal,
error recovery, the tool budget and graceful LLM outages; plus trace fields, report and hand-off schemas, the number
grounding check, the incorporation check, the cache and the follow-up memory.

In [18]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no", "."],
                   capture_output=True, text=True)
print(r.stdout[-1500:] or r.stderr[-1500:])

...........................                                              [100%]
27 passed in 1.37s



---
## Notes and limitations
- **Free-tier quota shapes the design:** 8k tokens/minute and 200k/day for `gpt-oss-120b`. Tool outputs are compact,
  calls are paced, and a daily-quota error falls back to `gpt-oss-20b` (logged) instead of failing.
- **Number grounding is checked, reasoning is not:** every figure in a report or hand-off must appear in a tool
  result (invented figures are sent back for removal), but a correct number can still be interpreted wrongly; an
  LLM-as-judge pass (as in Task 2) would be the next step.
- **News is not cross-checked against data:** a headline can contradict the price data (e.g. "enters bear market"
  while the price is near its 52-week high) and the writer may repeat it.
- **Live results vary between runs:** news, search results and the model's choices change; the outputs above are
  from the run saved in this notebook.
- Not investment advice.